# Hi-db remote index builder

This notebook builds the 256-shard `account_id` Parquet index and uploads it to the Hugging Face Storage Bucket configured for Hi-db.

**Before running:** use a Colab runtime with enough temporary disk for the full source scan and sorted index. Enter your Hugging Face write token only in the hidden prompt; do not paste it into notebook cells or commit it.

In [ ]:
import os, shutil
from getpass import getpass

print('Free disk space (GB):', round(shutil.disk_usage('/content').free / (1024**3), 2))
HF_TOKEN = getpass('Hugging Face write token (hidden input): ')
if not HF_TOKEN.startswith('hf_'):
    raise ValueError('That does not look like a Hugging Face token.')
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HI_DB_BUCKET'] = 'Chatpataprani/HITECH_DATABASE-bucket'
os.environ['HI_DB_INDEX_PREFIX'] = 'index/account_id'
print('Token stored in this runtime only; bucket:', os.environ['HI_DB_BUCKET'])

In [ ]:
!git clone https://github.com/chatpataprani/Hi-db.git /content/Hi-db
%cd /content/Hi-db
!pip -q install 'duckdb==1.2.2' 'huggingface_hub>=0.30.0'
print('Builder script ready.')

In [ ]:
# Run the one-time full-data scan, create sorted shards, and upload them.
# This can take a long time and needs enough temporary disk space.
!python scripts/build_remote_index.py

## After upload

The API expects shard files at `index/account_id/shard=00/data_0.parquet` through `shard=ff/data_0.parquet` (only non-empty shards are uploaded). Keep the Hugging Face token private. Then test the Vercel `/health` and `/lookup?account_id=...` endpoints.